In [1]:
#from flashtext import KeywordProcessor
from mstr_robotics.navigation import answer_prompts, mstr_objects
from mstr_robotics.report import rep, prompts
from mstrio.connection import Connection
from IPython.display import HTML
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path
import os
import json 

from mstr_robotics.user_RAG import keyword_processor, perplexity

u_keyword_processor=keyword_processor()
os_mcp_folder_str="C:\\coding\\python_io\\output_files\\MCP_data"

user_path="..\\config\\user_d.json"
with open(user_path, 'r') as file:
    user_d = json.load(file)

env_file="..\\config\\streamlit.env"
load_dotenv(env_file)
i_prompts=prompts()
i_rep=rep()
u_perplexity=perplexity()
i_mstr_objects=mstr_objects()

In [5]:
project_id = "B7CA92F04B9FAE8D941C3E9B7E0CD754"
conn_params =  user_d["conn_params"]
conn_params["project_id"]=project_id
conn = Connection(**conn_params)
conn.headers['Content-type'] = "application/json"

Connection to Strategy One Intelligence Server has been established.


In [6]:
# MSTR specitic objects and definitions
attribute_form_elements_df = pd.read_csv(os.path.join(os_mcp_folder_str, "attribute_form_elements.csv"))
attribute_elements_df = pd.read_csv(os.path.join(os_mcp_folder_str, "attribute_elements.csv"))
att_form_def_df = pd.read_csv(os.path.join(os_mcp_folder_str, "att_form_def.csv"))
obj_prp_rel_df = pd.read_csv(os.path.join(os_mcp_folder_str, "obj_prp_rel.csv"))
dos_rep_prp_rel_df = pd.read_csv(os.path.join(os_mcp_folder_str, "dos_rep_prp_rel.csv"))
dashboard_definitions_df = pd.read_csv(os.path.join(os_mcp_folder_str, "dashboard_definitions.csv"))
dashboard_chapter_filter_df = pd.read_csv(os.path.join(os_mcp_folder_str, "dashboard_chapter_filter.csv"))
dashboard_selector_filter_df = pd.read_csv(os.path.join(os_mcp_folder_str, "dashboard_selector_filter.csv"))

i_answer_prompts=answer_prompts(attribute_form_elements_df=attribute_form_elements_df
                                , attribute_elements_df=attribute_elements_df
                                , obj_prp_rel_df=obj_prp_rel_df
                                , att_form_def_df=att_form_def_df
                                , dos_rep_prp_rel_df=dos_rep_prp_rel_df
                                , dashboard_definitions_df=dashboard_definitions_df
                                , dashboard_chapter_filter_df=dashboard_chapter_filter_df
                                , dashboard_selector_filter_df=dashboard_selector_filter_df
                                )

In [7]:
# tool agnostic definitions. Only MSTR for the moment
element_df_d_l=[{"df":attribute_form_elements_df,"key_col":"element_val","key_type":"element_val","rag_cols": ["attribute_name", "form_name", "element_val"]},
                 {"df":attribute_elements_df,"key_col":"element_val","key_type":"element_val","rag_cols": ["attribute_name", "element_val"]}
                ]

bi_obj_df=obj_prp_rel_df[["object_name", "obj_type", "object_id"]][obj_prp_rel_df["obj_type"].isin(["attribute","metric"]) ]

obj_df_d_l=[{"df":bi_obj_df,"key_col":"object_name","key_type":"object_name","rag_cols": ["object_name", "obj_type"]}]
            


In [8]:
msg_t="Please show me the Cost_1, Revenue and Profit for the attributes Year,Category, Region"
msg_t= msg_t + " and filter for the yaer 2021, 2022 and 2023"
msg_t= msg_t + " and Categories starting with B"
msg_t= msg_t + " and Revenue is between 10 and 1000000"


key_word_l=u_keyword_processor.extract_keywords(msg_t=msg_t)

att_elem_str=i_mstr_objects.get_att_elem_str(element_df_d_l, key_word_l=key_word_l)
bi_obj_str=i_mstr_objects.get_att_elem_str(obj_df_d_l, key_word_l=key_word_l)

sys_cont=u_perplexity.rag_sys_cont(key_word_l=key_word_l,att_elem_str=att_elem_str, bi_obj_str=bi_obj_str)

message_check_d={}
message_check_d_l=[]
message_check_d["msg_nr"] = "1"
message_check_d["msg_t"] = msg_t
message_check_d=u_perplexity.call_perplexity( msg_t=msg_t, sys_cont=sys_cont, message_check_d=message_check_d, temperature=0.1)
message_check_d_l.append(message_check_d.copy())

#message_check_d["step"] = "template"
#message_check_d = u_perplexity.call_perplexity(msg_t=msg_t,message_check_d=message_check_d, sys_cont=sys_cont)
#message_check_d_l.append(message_check_d.copy())
bi_request_d=u_perplexity.parse_and_structure(message_check_d_l)
bi_request_d

{'msg_nr': '1',
 'msg_t': 'Please show me the Cost_1, Revenue and Profit for the attributes Year,Category, Region and filter for the yaer 2021, 2022 and 2023 and Categories starting with B and Revenue is between 10 and 1000000',
 'attributes': ['Year', 'Category', 'Region'],
 'metrics': ['Cost_1', 'Revenue', 'Profit'],
 'filter': {'att_qual_1': {'attribute': 'Category',
   'Column': 'desc',
   'operator': 'BeginsWith',
   'value': 'B'},
  'att_element_1': {'attribute': 'Year',
   'operator': 'In',
   'element_list': ['2021', '2022', '2023']},
  'metric_filter_1': {'level': None,
   'metric': 'Revenue',
   'operator': 'Between',
   'value': [10, 1000000]}},
 'question': ''}

In [9]:
report_id="25D40AD444B6D51B333021ADFB219501"

prompt_answ=i_answer_prompts.AI_mstr_prp_page_ans( vector_store=u_keyword_processor
                                                  ,bi_request_d=bi_request_d
                                                  ,rep_dos_id=report_id
                                                  )
prompt_answ


att_qual_1
att_elemen
metric_fil


'{"prompts": [{"id": "1F21042E4FC08886A9D7E4920CA4EC59", "type": "EXPRESSION", "answers": {"expression": {"operator": "And", "operands": [{"operator": "AND", "operands": [{"operator": "BeginsWith", "operands": [{"type": "form", "attribute": {"id": "8D679D3711D3E4981000E787EC6DE8A4"}, "form": {"id": "CCFBE2A5EADB4F50941FB879CCF1721C"}}, {"type": "constant", "dataType": "Char", "value": "B"}]}, {"operator": "In", "operands": [{"type": "form", "attribute": {"id": "8D679D5111D3E4981000E787EC6DE8A4"}, "form": {"id": "45C11FA478E745FEA08D781CEA190FE5"}}, {"type": "constants", "dataType": "Numeric", "values": ["2021", "2022", "2023"]}]}]}]}}}, {"id": "14B57F1F46127BC6E6F746888BC3C7CB", "type": "EXPRESSION", "answers": {"expression": {"operator": "AND", "operands": [{"operator": "Between", "operands": [{"type": "metric", "id": "4C05177011D3E877C000B3B2D86C964F"}, {"type": "constant", "dataType": "Real", "value": "10"}, {"type": "constant", "dataType": "Real", "value": "1000000"}], "level": {"t

In [10]:
ai_rep_name="dyn_prompt_page_botstat"
ai_rep_folder_id="2F2302AE4D1C2DDDFA9CDCB46802B185"
report_id="25D40AD444B6D51B333021ADFB219501"

rep_id=i_answer_prompts.save_AI_rep(conn=conn,report_id=report_id
                                  ,prompt_answ=prompt_answ
                                  ,ai_rep_name=ai_rep_name
                                  ,ai_rep_folder_id=ai_rep_folder_id)
new_rep_id=rep_id.json()["id"]
instance_id = i_rep.open_Instance(conn=conn, report_id=new_rep_id)
df=i_rep.report_df(conn=conn, report_id=new_rep_id, instance_id=instance_id)
df

Report object named: 'dyn_prompt_page_botstat' with ID: 'C731B12148180FDA527FB5BCE95E618E'


,Category,Year,Region,Revenue,Profit
0,Books,2021,Central,124045.80,26768.360
1,Books,2021,Mid-Atlantic,114815.95,24850.750
2,Books,2021,Northeast,218225.70,47111.068
3,Books,2021,Northwest,45521.65,9961.775
4,Books,2021,South,133762.05,28260.394
5,Books,2021,Southeast,56494.00,12099.871
6,Books,2021,Southwest,95810.20,20854.605
7,Books,2021,Web,79531.15,17120.360
8,Books,2022,Central,154588.50,33372.717
9,Books,2022,Mid-Atlantic,136809.85,29539.700


In [11]:
ai_rep_name="dyn_prompt_page_botstat"
report_id="25D40AD444B6D51B333021ADFB219501"
#report_id="29FBB4884282ED7203AFD2B247C57688"

rep_id=i_answer_prompts.save_AI_rep(conn=conn,report_id=report_id
                                  ,prompt_answ=prompt_answ
                                  ,ai_rep_name=ai_rep_name
                                  ,promptOption ="filterAndTemplate"
                                  ,ai_rep_folder_id=ai_rep_folder_id)
new_rep_id=rep_id.json()["id"]
link=i_rep.web_base_url(conn=conn,report_id=new_rep_id)
HTML(link)

http://217.154.213.84:8080/MicroStrategy/servlet/mstrWeb?Server=217.154.213.84&Project=MicroStrategy+Tutorial&evt=4001&src=mstrWeb.4001&reportViewMode=1&reportID=C731B12148180FDA527FB5BCE95E618E&currentViewMedia=2


In [ ]:
f="C:\\Users\danie\\Downloads\\display_difference_dash.xlsx"
df = pd.read_excel(f)
result = df.groupby('org_obj_key')['json_key_path'].apply(list).reset_index()
d_l=result.to_dict


In [ ]:
d_l=result.to_dict('records')
d_l